# Phase 7: Export Assets for the Streamlit Demo

Packs everything the demo app needs into one small folder: the 10 trained CNNs (weights only), the decision threshold and calibration, the PTB-XL normalization statistics, and 20 example ECGs from the test set.

| Output | Size | Used for |
|---|---|---|
| `cnn_binary_s0-4.npz` | about 1.6 MB each | Normal vs Abnormal (Task A) |
| `cnn_multi_s0-4.npz` | about 1.6 MB each | 5 diagnostic classes (Task B) |
| `config.json` | 1 KB | threshold, Platt scaling, normalization, lead and class names |
| `examples.npz` | about 1 MB | 20 test-set ECGs to try in the app |
| `example_upload_100Hz.csv` | about 150 KB | a ready-made file to test the upload option |

Runtime: **CPU is enough** (about 3 minutes). Run all cells in order.

## Step 1: Environment Setup

In [ ]:
# Step 1: Mount Drive, make src/ importable, and define project paths
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, shutil
PROJECT = '/content/drive/MyDrive/ecg-project'
OUT, MODEL_DIR = f'{PROJECT}/data_processed', f'{PROJECT}/models'
ASSETS = f'{PROJECT}/app_assets'
os.makedirs(ASSETS, exist_ok=True)
sys.path.append(f'{PROJECT}/src')

import numpy as np, pandas as pd
import keras
import data, models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
THR = 0.462                                   # CNN threshold chosen on validation in Phase 3

Connects to Drive and creates the folder `ecg-project/app_assets`, where every file for the app is saved.

## Step 2: Load Data and Models

In [ ]:
# Step 2: Load validation and test data, metadata and the 10 trained CNNs; confirm the Phase 3 AUROCs
X_val, yb_val = data.load_split('val')
X_test, yb_test = data.load_split('test')
ym_test = np.load(f'{OUT}/Ymulti_test.npy')
meta = pd.read_csv(f'{OUT}/metadata.csv', index_col='ecg_id')
tm = meta[meta['split'] == 'test'].copy()

cnn_bin = [keras.models.load_model(f'{MODEL_DIR}/cnn_binary_s{s}.keras') for s in range(5)]
cnn_multi = [keras.models.load_model(f'{MODEL_DIR}/cnn_multi_s{s}.keras') for s in range(5)]

def ens_predict(ms, X):
    return np.mean([m.predict(X, batch_size=512, verbose=0) for m in ms], axis=0)

p_val = ens_predict(cnn_bin, X_val).ravel()
p_test = ens_predict(cnn_bin, X_test).ravel()
pm_test = ens_predict(cnn_multi, X_test)
print(f'Task A AUROC: {roc_auc_score(yb_test, p_test):.3f} (expected 0.943)')
print(f"Task B macro AUROC: {roc_auc_score(ym_test, pm_test, average='macro'):.3f} (expected 0.933)")
print('Metadata columns:', list(tm.columns))

Reloads the final models and re-checks their test AUROC, so the app is guaranteed to use the same models as the results in the README.

## Step 3: Save Model Weights

In [ ]:
# Step 3: Save each CNN's weights as a plain list of arrays (small, and loads in any TensorFlow/Keras version)
for task, ms in [('binary', cnn_bin), ('multi', cnn_multi)]:
    for s, m in enumerate(ms):
        np.savez_compressed(f'{ASSETS}/cnn_{task}_s{s}.npz', *m.get_weights())

# Check: rebuild one model from the saved weights and confirm it gives identical predictions
w = np.load(f'{ASSETS}/cnn_binary_s0.npz')
check = models.build_cnn(1)
check.set_weights([w[f'arr_{i}'] for i in range(len(w.files))])
diff = np.abs(check.predict(X_test[:64], verbose=0) - cnn_bin[0].predict(X_test[:64], verbose=0)).max()
print(f'Max difference after reload: {diff:.2e} (should be about 0)')

Saves only the learned numbers (weights), not the training state, so each model shrinks to about 1.6 MB. The app rebuilds the same CNN architecture and loads these weights, which avoids Keras version problems on the hosting server. The check confirms that the reloaded model predicts exactly as the original.

## Step 4: Calibration and Configuration

In [ ]:
# Step 4: Fit Platt scaling on validation predictions and save threshold, calibration and normalization settings
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
platt = LogisticRegression(C=1e6).fit(logit(p_val)[:, None], yb_val)
a, b = float(platt.coef_[0, 0]), float(platt.intercept_[0])
calibrate = lambda p: 1 / (1 + np.exp(-(a * logit(p) + b)))

MU, SD = np.load(f'{OUT}/norm_stats.npy')
config = {
    'leads': LEADS, 'classes': CLASSES, 'fs': 100, 'seconds': 10,
    'threshold_raw': THR, 'threshold_calibrated': float(calibrate(np.array(THR))),
    'platt_a': a, 'platt_b': b,
    'mu': np.asarray(MU, float).reshape(-1).tolist(), 'sd': np.asarray(SD, float).reshape(-1).tolist(),
    'test_auroc_task_a': round(float(roc_auc_score(yb_test, p_test)), 3),
}
json.dump(config, open(f'{ASSETS}/config.json', 'w'), indent=1)
print(f"Platt: a = {a:.3f}, b = {b:.3f} | threshold {THR} = calibrated probability {config['threshold_calibrated']:.3f}")

The app shows a **calibrated** probability (Platt scaling, fitted on validation as in Phase 3), so "70% abnormal" means roughly 70 in 100 such ECGs are abnormal. The decision itself still uses the validation-chosen threshold of 0.462; because Platt scaling never changes the ranking, this is the same decision expressed on the calibrated scale.

## Step 5: Choose 20 Example ECGs

In [ ]:
# Step 5: Pick 20 test ECGs covering every diagnosis, mixed findings, model mistakes and young patients
rng = np.random.default_rng(7)
sc = tm['superclass'].fillna('').str.split('|')
n_labels = sc.str.len().values
age = tm['age'].values
pred_abn = p_test >= THR
picks = []

def pick(mask, n, category):
    idx = [i for i in np.where(mask)[0] if i not in [p for p, _ in picks]]
    for i in rng.choice(idx, size=min(n, len(idx)), replace=False):
        picks.append((int(i), category))

pick((yb_test == 0) & ~pred_abn, 3, 'Normal')
for k, c in enumerate(CLASSES[1:], start=1):
    pick((n_labels == 1) & (ym_test[:, k] == 1) & pred_abn & (pm_test[:, k] > 0.5), 2, c)
pick((n_labels >= 2) & pred_abn, 2, 'Several findings')
pick((yb_test == 1) & ~pred_abn & (age >= 40), 2, 'Missed by the model')
pick((yb_test == 0) & pred_abn, 2, 'False alarm')
pick((yb_test == 1) & (age < 40) & pred_abn, 2, 'Under 40')
pick((yb_test == 1) & (age < 40) & ~pred_abn, 1, 'Under 40, missed')

idx = np.array([i for i, _ in picks])
sex = tm['sex'].values[idx]
sex = np.array(['Female' if str(s).strip() in ('1', '1.0', 'F', 'female') else 'Male' for s in sex])
ex_age = np.array(['89+' if a >= 300 else str(int(a)) for a in age[idx]])
np.savez_compressed(f'{ASSETS}/examples.npz',
                    X=X_test[idx].astype(np.float32), ecg_id=tm.index.values[idx],
                    category=np.array([c for _, c in picks]), superclass=np.array([str(v) for v in tm['superclass'].fillna('').values[idx]], dtype=str),
                    y=yb_test[idx].astype(int), Y=ym_test[idx].astype(int), age=ex_age, sex=sex)

print(pd.DataFrame({'ECG': tm.index.values[idx], 'category': [c for _, c in picks], 'label': tm['superclass'].values[idx],
                    'age': ex_age, 'sex': sex, 'p_abnormal': p_test[idx].round(2)}).to_string(index=False))

The examples are chosen at random (fixed seed) within each group, not cherry-picked, and the set deliberately includes the model's mistakes (a missed abnormal ECG, a false alarm, a missed case under 40) so the demo shows its limits honestly. PTB-XL records ages above 89 as 300 for privacy; they are shown as "89+".

## Step 6: Example File for the Upload Option

In [ ]:
# Step 6: Write one example ECG as a CSV in millivolts (12 lead columns, 100 Hz) to test the app's upload option
k = [c for _, c in picks].index('CD') if 'CD' in [c for _, c in picks] else 0
ecg_mv = X_test[idx[k]] * np.asarray(SD).reshape(1, -1) + np.asarray(MU).reshape(1, -1)
pd.DataFrame(ecg_mv.round(4), columns=LEADS).to_csv(f'{ASSETS}/example_upload_100Hz.csv', index=False)
print(f'Saved ECG {tm.index.values[idx[k]]} ({picks[k][1]}) as example_upload_100Hz.csv')

Undoes the standardization to get millivolts, the unit a real ECG export would use. Uploading this file in the app should give almost the same prediction as picking the same ECG from the example list (not identical, because the app filters every upload and this file is already filtered).

## Step 7: Package for Download

In [ ]:
# Step 7: List the files with their sizes and zip the folder for download
total = 0
for f in sorted(os.listdir(ASSETS)):
    size = os.path.getsize(f'{ASSETS}/{f}') / 1e6
    total += size
    print(f'{f:30s} {size:6.2f} MB')
print(f'{"Total":30s} {total:6.2f} MB')
shutil.make_archive(f'{PROJECT}/app_assets', 'zip', ASSETS)
print(f'Download {PROJECT}/app_assets.zip and unzip it into the repo folder app/assets/')

The whole folder is about 20 MB, well under GitHub's 100 MB file limit. Download `app_assets.zip` from `ecg-project` on Google Drive and unzip it into `app/assets/` in your repo.